In [ ]:
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from collections import defaultdict
from itertools import combinations

def plot_correlations(train_df,feature_indices):
    X_train_reduced = train_df.iloc[:,feature_indices]
    
    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(25, 8))
    corr = spearmanr(X_train_reduced).correlation

    # Ensure the correlation matrix is symmetric
    corr = (corr + corr.T) / 2
    np.fill_diagonal(corr, 1)

    # We convert the correlation matrix to a distance matrix before performing
    # hierarchical clustering using Ward's linkage.
    distance_matrix = 1 - np.abs(corr)
    dist_linkage = hierarchy.ward(squareform(distance_matrix))
    dendro = hierarchy.dendrogram(
        dist_linkage, labels=X_train_reduced.columns.to_list(), ax=ax1, leaf_rotation=90
    )

    dendro_idx = np.arange(0, len(dendro["ivl"]))

    ax2.imshow(corr[dendro["leaves"], :][:, dendro["leaves"]])
    ax2.set_xticks(dendro_idx)
    ax2.set_yticks(dendro_idx)
    ax2.set_xticklabels(dendro["ivl"], rotation="vertical")
    ax2.set_yticklabels(dendro["ivl"])
    _ = fig.tight_layout()


    plt.show()

def num_features_per_threshold(train_df,thresholds):
    corr = spearmanr(train_df).correlation

    # Ensure the correlation matrix is symmetric
    corr = (corr + corr.T) / 2
    np.fill_diagonal(corr, 1)

    # convert the correlation matrix to a distance matrix before performing
    # hierarchical clustering using Ward's linkage.
    distance_matrix = 1 - np.abs(corr)
    dist_linkage = hierarchy.ward(squareform(distance_matrix))

    for threshold in thresholds:
        cluster_ids = hierarchy.fcluster(dist_linkage, threshold, criterion="distance")
        n_features = len(np.unique(cluster_ids))
        print(f"threshold {threshold}: {n_features} features")


def get_uncorrelated_features(train_df,threshold):

    # correlation among all variables
    corr = spearmanr(train_df).correlation
    corr = (corr + corr.T) / 2
    np.fill_diagonal(corr, 1)

    # convert correlation to distance
    distance_matrix = 1 - np.abs(corr)
    # clustering algorithm
    dist_linkage = hierarchy.ward(squareform(distance_matrix))

    cluster_ids = hierarchy.fcluster(dist_linkage, threshold, criterion="distance")
    cluster_id_to_feature_ids = defaultdict(list)
    for idx, cluster_id in enumerate(cluster_ids):
        cluster_id_to_feature_ids[cluster_id].append(idx)

    # select feature from each cluster with highest correlation to % tree canopy
    selected_features = []
    for cluster_features in cluster_id_to_feature_ids.values():
        corr_to_y = [spearmanr(X_train.iloc[:,i],y_train).statistic for i in cluster_features]
        best = cluster_features[np.argmax(np.abs(corr_to_y))]
        selected_features.append(best)

    print(f'selected features: {train_df.columns[selected_features]}')

    return selected_features

def filter_high_correlations(train_df,selected_features,corr_threshold):
        
        to_remove = set()
        for f1, f2 in combinations(list(selected_features),2):
            if f1 in to_remove or f2 in to_remove:
                continue
            r, _ = spearmanr(train_df.iloc[:,f1],train_df.iloc[:,f2])

            if abs(r) > corr_threshold:
                r1 = abs(spearmanr(X_train.iloc[:,f1],y_train)[0])
                r2 = abs(spearmanr(X_train.iloc[:,f2],y_train)[0])

                to_remove.add(f1 if r1 < r2 else f2)
            
        final_features = [f for f in selected_features if f not in to_remove]
        print(f'removed {len(to_remove)}: {train_df.columns[list(to_remove)]}')
        
        return final_features

In [ ]:
## read in data
train_df = pd.read_parquet(root / 'data' / 'nyc21' / 'test_df_clf.parquet')
test_df = pd.read_parquet(root / 'data' / 'nyc21' / 'train_df_clf.parquet')

feature_cols = [c for c in train_df.columns if c.startswith('band_')]

X_train = train_df[feature_cols]
y_train = train_df['canopy_binary']
X_test = test_df[feature_cols]
y_test = test_df['canopy_binary']

In [ ]:
## how many features will be retained at each threshold?
num_features_per_threshold(X_train)

In [ ]:
selected_features1 = get_uncorrelated_features(X_train,threshold=1)
selected_features15 = get_uncorrelated_features(X_train,threshold=1.5)
selected_features2 = get_uncorrelated_features(X_train, threshold=2)
selected_features25 = get_uncorrelated_features(X_train,threshold=2.5)
selected_features3 = get_uncorrelated_features(X_train,threshold=3)

In [ ]:
### filter any variables with correlation greater than threshold
# retain the one with higher correlation with y
final_features1 = filter_high_correlations(X_train,selected_features=selected_features1,corr_threshold=0.7)

In [ ]:
#### plot baseline scores with increasing number of features
feature_sets = [final_features1,final_features15,final_features2,final_features25,final_features3]

def test_features_clf(feature_sets,X_train,X_test,y_train,y_test):
    auc_scores = []
    num_features = []
    for features in feature_sets:
        X_train_reduced = X_train.iloc[:,features]
        X_test_reduced = X_test.iloc[:,features]
        
        # fit model for permutation imp
        rf_explore = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        )
        rf_explore.fit(X_train_reduced, y_train)

        baseline_auc = roc_auc_score(y_test, rf_explore.predict_proba(X_test_reduced)[:, 1])
        auc_scores.append(baseline_auc)
        num_features.append(len(features))

    out_df = pd.DataFrame({'num_features':num_features,'auc_scores':auc_scores})

    return out_df




In [ ]:
ax = out_df.plot(x='num_features',y='auc_scores',legend=False)
ax.set_ylim((0.5,1))
ax.set_xlabel('Number of Features Included')
ax.set_ylabel('AUC Score')
ax.set_title('AUC Scores with More Correlated Features Included')

plt.show()

# for the classifier, scores barely change with fewer features

In [ ]:
## write the feature set we want to use to a file
r = X_train.iloc[:,final_features25]
clf_features = list(r.columns)

import json
import os
#fp = root / 'data' / 'nyc21' / 'clf_features.json'

with open(fp,'w') as f:
    json.dump(clf_features,f)